<a href="https://colab.research.google.com/github/hgshreyas/BIS_LAB/blob/main/1BM24CS108_Public_Transport_PSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import random

# --- 1. User Inputs & Configuration with Defaults ---
try:
    # Try to get user input, or fall back to defaults if in non-interactive environment
    n_buses_str = input("Enter number of buses: ").strip()
    n = int(n_buses_str) if n_buses_str else 3

    passengers_str = input("Enter passenger arrival times in minutes, space-separated: ").strip()
    if passengers_str:
        passengers = list(map(int, passengers_str.split()))
    else:
        passengers = [10, 15, 25, 40]

    start_str = input("Enter operating start time in minutes: ").strip()
    start = int(start_str) if start_str else 0

    end_str = input("Enter operating end time in minutes: ").strip()
    end = int(end_str) if end_str else 60
except (ValueError, OSError):
    # Default values for quick execution
    n = 3
    passengers = [10, 15, 25, 40]
    start = 0
    end = 60

print(f"\n--- Running PSO with Configuration ---")
print(f"Number of buses: {n}")
print(f"Passenger arrivals: {passengers}")
print(f"Operation window: {start} to {end} minutes\n")

# --- 2. PSO Parameters ---
N = 20        # Number of particles
T = 50        # Maximum iterations
w = 0.5       # Inertia weight
c1 = 1.5      # Cognitive coefficient
c2 = 1.5      # Social coefficient

# --- 3. Fitness Function ---
# Calculates total passenger waiting time
def fitness(schedule):
    total_waiting_time = 0
    for p in passengers:
        # Find all scheduled buses departing after or at passenger arrival
        valid_buses = [bus for bus in schedule if bus >= p]
        if valid_buses:
            total_waiting_time += min(valid_buses) - p
        else:
            # Penalty if a passenger misses all buses (waiting until end of shift)
            total_waiting_time += end - p
    return total_waiting_time

# --- 4. Initialization ---
# Initialize particle positions (sorted departure schedules)
X = [sorted(random.uniform(start, end) for _ in range(n)) for _ in range(N)]

# Initialize particle velocities
V = [[random.uniform(-2, 2) for _ in range(n)] for _ in range(N)]

# Initialize personal best positions
pBest = [x[:] for x in X]

# Initialize global best position
gBest = min(pBest, key=fitness)[:]

# --- 5. PSO Optimization Loop ---
for iteration in range(T):
    for i in range(N):
        for j in range(n):
            r1 = random.random()
            r2 = random.random()

            # Velocity update equation
            V[i][j] = (
                w * V[i][j]
                + c1 * r1 * (pBest[i][j] - X[i][j])
                + c2 * r2 * (gBest[j] - X[i][j])
            )

            # Position update equation
            X[i][j] += V[i][j]

            # Boundary constraints: Keep bus departures within scheduled hours
            X[i][j] = max(start, min(end, X[i][j]))

        # Sort departure times for logical chronological consistency
        X[i].sort()

        # Evaluate fitness and update personal best
        if fitness(X[i]) < fitness(pBest[i]):
            pBest[i] = X[i][:]

        # Evaluate fitness and update global best
        if fitness(X[i]) < fitness(gBest):
            gBest = X[i][:]

    # Optional: print convergence progress every 10 iterations
    if (iteration + 1) % 10 == 0 or iteration == 0:
        print(f"Iteration {iteration + 1}/{T} | Current Best Waiting Time: {round(fitness(gBest), 2)} mins")

# --- 6. Results ---
print("\n--- Optimization Results ---")
print("Optimal Bus Departure Schedule (Minutes from start):", [round(bus_time, 2) for bus_time in gBest])
print("Minimum Total Passenger Waiting Time:", round(fitness(gBest), 2), "minutes")

Enter number of buses: 3
Enter passenger arrival times in minutes, space-separated: 10 20 35 50
Enter operating start time in minutes: 0
Enter operating end time in minutes: 60

--- Running PSO with Configuration ---
Number of buses: 3
Passenger arrivals: [10, 20, 35, 50]
Operation window: 0 to 60 minutes

Iteration 1/50 | Current Best Waiting Time: 27.28 mins
Iteration 10/50 | Current Best Waiting Time: 11.02 mins
Iteration 20/50 | Current Best Waiting Time: 10.2 mins
Iteration 30/50 | Current Best Waiting Time: 10.03 mins
Iteration 40/50 | Current Best Waiting Time: 10.01 mins
Iteration 50/50 | Current Best Waiting Time: 10.0 mins

--- Optimization Results ---
Optimal Bus Departure Schedule (Minutes from start): [20.0, 35.0, 50.0]
Minimum Total Passenger Waiting Time: 10.0 minutes
